<a href="https://colab.research.google.com/github/HopeSilkina/deposits_forecast_project/blob/main/notebooks/05_SARIMA_Modeling_Deposits_Forecast_Ru.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# БЛОК 5: МОДЕЛИ ВРЕМЕННЫХ РЯДОВ (SARIMA)
# Проект: Прогнозирование объема вкладов населения РФ
# Автор: Надежда Силкина
# Дата: 2026
# ============================================================

# ============================================================
# 1. ПОДКЛЮЧЕНИЕ БИБЛИОТЕК
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Библиотеки загружены")

# ============================================================
# 2. ЗАГРУЗКА ДАННЫХ
# ============================================================

url = 'https://raw.githubusercontent.com/HopeSilkina/deposits_forecast_project/main/data/processed_deposits_data.xlsx'
df = pd.read_excel(url, sheet_name='data')
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"✅ Данные загружены. Записей: {len(df)}")
print(f"Период: с {df.index.min()} по {df.index.max()}")

# ============================================================
# 3. МЕТОДОЛОГИЧЕСКОЕ ОБОСНОВАНИЕ ВЫБОРА SARIMA
# ============================================================

print("\n" + "="*60)
print("3. ПОЧЕМУ SARIMA, А НЕ PROPHET")
print("="*60)

print("""
📌 КЛЮЧЕВАЯ ПРОБЛЕМА: АВТОКОРРЕЛЯЦИЯ ОСТАТКОВ 4-ГО ПОРЯДКА

Во всех предыдущих моделях (базовая Ridge, полная модель с 38 признаками)
диагностика остатков на обучающей выборке выявила значимую автокорреляцию
4-го порядка (тест Бройша-Годфри, p < 0.05).

ПОЧЕМУ НЕ PROPHET:
- Prophet не учитывает автокорреляцию в остатках напрямую
- При автокоррелированных ошибках дает смещенные прогнозы

ПОЧЕМУ SARIMA:
- Явно моделирует автокорреляцию через AR/MA компоненты
- Сезонные компоненты могут учесть квартальную/годовую структуру
- Ljung-Box тест позволяет проверить, устранена ли автокорреляция

ВЫВОД: Выбираем SARIMA для решения проблемы автокорреляции.
""")

# ============================================================
# 4. АНАЛИЗ СТАЦИОНАРНОСТИ РЯДА DEPOS
# ============================================================

print("\n" + "="*60)
print("4. АНАЛИЗ СТАЦИОНАРНОСТИ РЯДА DEPOS")
print("="*60)

print("""
📌 ПРОВЕРЯЕМЫЙ РЯД: DEPOS (объем вкладов населения РФ, млрд руб.)

Два теста с разными гипотезами:
- ADF: H₀ = ряд НЕстационарный (есть единичный корень)
- KPSS: H₀ = ряд стационарный (нет единичного корня)
""")

# 4.1. Тест ADF (константа, без тренда)
print("\n🔍 Тест Дики-Фуллера (ADF):")
print("   H₀: ряд нестационарный | H₁: ряд стационарный")
adf_result = adfuller(df['DEPOS'], autolag='AIC', regression='c')
print(f"   ADF-статистика: {adf_result[0]:.4f}")
print(f"   p-значение: {adf_result[1]:.4f}")
print(f"   Критические значения: 1%: {adf_result[4]['1%']:.4f}, 5%: {adf_result[4]['5%']:.4f}")
print(f"   Вывод: {'Стационарный ✅' if adf_result[1] < 0.05 else 'Нестационарный ❌ (p > 0.05, не отвергаем H₀)'}")

# 4.2. Тест KPSS (константа + тренд)
print("\n🔍 Тест KPSS:")
print("   H₀: ряд стационарный | H₁: ряд нестационарный")
kpss_result = kpss(df['DEPOS'], regression='ct')
print(f"   KPSS-статистика: {kpss_result[0]:.4f}")
print(f"   p-значение: {kpss_result[1]:.4f}")
print(f"   Критические значения: 1%: {kpss_result[3]['1%']:.4f}, 5%: {kpss_result[3]['5%']:.4f}")
print(f"   Вывод: {'Стационарный ✅' if kpss_result[1] > 0.05 else 'Нестационарный ❌ (p < 0.05, отвергаем H₀)'}")

print("\n📌 ОБЩИЙ ВЫВОД: Оба теста подтверждают, что ряд DEPOS НЕСТАЦИОНАРНЫЙ")
print("   → Требуется дифференцирование")

# 4.3. График ряда
plt.figure(figsize=(14, 6))
plt.plot(df.index, df['DEPOS'], linewidth=2, color='steelblue')
plt.title('Ряд DEPOS: Объем вкладов населения РФ', fontsize=14)
plt.xlabel('Дата')
plt.ylabel('млрд руб.')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_depos_series.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 5. ОПРЕДЕЛЕНИЕ ПОРЯДКА ДИФФЕРЕНЦИРОВАНИЯ (d)
# ============================================================

print("\n" + "="*60)
print("5. ОПРЕДЕЛЕНИЕ ПОРЯДКА ДИФФЕРЕНЦИРОВАНИЯ (d)")
print("="*60)

print("""
📌 МЕТОДИКА: Последовательно проверяем разности ряда на стационарность.
d=0 → если нестационарен, d=1 → если нестационарен, d=2.
""")

# --- Проверка первой разности (d=1) ---
print("\n🔍 Проверка ПЕРВОЙ разности (d=1):")
df['DEPOS_diff1'] = df['DEPOS'].diff()

adf_diff1 = adfuller(df['DEPOS_diff1'].dropna(), autolag='AIC', regression='c')
kpss_diff1 = kpss(df['DEPOS_diff1'].dropna(), regression='c')

print(f"   ADF: stat = {adf_diff1[0]:.4f}, p = {adf_diff1[1]:.4f}")
print(f"   KPSS: stat = {kpss_diff1[0]:.4f}, p = {kpss_diff1[1]:.4f}")

d1_stationary = (adf_diff1[1] < 0.05) and (kpss_diff1[1] > 0.05)

if d1_stationary:
    print(f"   ✅ Первая разность СТАЦИОНАРНА → d = 1")
    d_order = 1
else:
    print(f"   ❌ Первая разность НЕСТАЦИОНАРНА (ADF p={adf_diff1[1]:.4f} > 0.05)")
    print(f"   → Проверяем вторую разность")

    # --- Проверка второй разности (d=2) ---
    print("\n🔍 Проверка ВТОРОЙ разности (d=2):")
    df['DEPOS_diff2'] = df['DEPOS_diff1'].diff()

    adf_diff2 = adfuller(df['DEPOS_diff2'].dropna(), autolag='AIC', regression='c')
    kpss_diff2 = kpss(df['DEPOS_diff2'].dropna(), regression='c')

    print(f"   ADF: stat = {adf_diff2[0]:.4f}, p = {adf_diff2[1]:.4f}")
    print(f"   KPSS: stat = {kpss_diff2[0]:.4f}, p = {kpss_diff2[1]:.4f}")

    d2_stationary = (adf_diff2[1] < 0.05) and (kpss_diff2[1] > 0.05)

    if d2_stationary:
        print(f"   ✅ Вторая разность СТАЦИОНАРНА → d = 2")
        d_order = 2
    else:
        print(f"   ⚠️ Даже вторая разность нестационарна (необычно для экономических рядов)")
        d_order = 2  # обычно достаточно

# Графики разностей
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(df.index, df['DEPOS'], linewidth=1.5, color='steelblue')
axes[0].set_title('Исходный ряд (d=0)')
axes[0].set_ylabel('DEPOS')
axes[0].grid(True, alpha=0.3)

axes[1].plot(df.index, df['DEPOS_diff1'], linewidth=1.5, color='darkorange')
axes[1].set_title(f'Первая разность (d=1): ADF p={adf_diff1[1]:.4f}, KPSS p={kpss_diff1[1]:.4f}')
axes[1].set_ylabel('Δ DEPOS')
axes[1].grid(True, alpha=0.3)

if d_order == 2:
    axes[2].plot(df.index, df['DEPOS_diff2'], linewidth=1.5, color='green')
    axes[2].set_title(f'Вторая разность (d=2): ADF p={adf_diff2[1]:.4f}, KPSS p={kpss_diff2[1]:.4f}')
    axes[2].set_ylabel('Δ² DEPOS')
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].axis('off')

plt.tight_layout()
plt.savefig('05_diff_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n📊 ИТОГ: Выбран порядок дифференцирования d = {d_order}")

# ============================================================
# 6. АНАЛИЗ ACF И PACF
# ============================================================

print("\n" + "="*60)
print("6. АНАЛИЗ ACF И PACF")
print("="*60)

print("""
📌 КАК ЧИТАТЬ ГРАФИКИ ACF/PACF:
- Затененная область — 95% доверительный интервал (±1.96/√n)
- Столбец, выходящий за область → автокорреляция значима (p < 0.05)
- Лаг 0 всегда равен 1 (ряд с самим собой) — не учитываем
- Лаг 1 — это ПЕРВЫЙ столбец после лага 0
""")

# Создаем обучающую выборку (последние 12 месяцев - тест)
train_size = len(df) - 12
depos_train = df['DEPOS'].iloc[:train_size]
depos_test = df['DEPOS'].iloc[train_size:]

print(f"\n📊 Обучающая выборка: {len(depos_train)} записей")
print(f"📊 Тестовая выборка: {len(depos_test)} записей")
print(f"📊 Период теста: {depos_test.index[0]} — {depos_test.index[-1]}")

# Дифференцированный ряд (d=2)
depos_train_diff = depos_train.diff().diff().dropna()

# Графики ACF и PACF с добавленными линиями значимости
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Порог значимости (95% ДИ)
n_obs = len(depos_train)
threshold = 1.96 / np.sqrt(n_obs)

# ACF исходного ряда
plot_acf(depos_train, lags=36, ax=axes[0, 0])
axes[0, 0].axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'±1.96/√n = ±{threshold:.3f}')
axes[0, 0].axhline(y=-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[0, 0].set_title('ACF: Исходный ряд (DEPOS, d=0)')
axes[0, 0].set_xlabel('Лаг (месяцы)')
axes[0, 0].set_ylabel('Автокорреляция')
axes[0, 0].legend(fontsize=8)

# PACF исходного ряда
plot_pacf(depos_train, lags=36, ax=axes[0, 1])
axes[0, 1].axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'±1.96/√n')
axes[0, 1].axhline(y=-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[0, 1].set_title('PACF: Исходный ряд (DEPOS, d=0)')
axes[0, 1].set_xlabel('Лаг (месяцы)')
axes[0, 1].set_ylabel('Частичная автокорреляция')
axes[0, 1].legend(fontsize=8)

# ACF дифференцированного ряда (d=2)
plot_acf(depos_train_diff, lags=36, ax=axes[1, 0])
axes[1, 0].axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'±1.96/√n')
axes[1, 0].axhline(y=-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[1, 0].set_title('ACF: Дифференцированный ряд (d=2)')
axes[1, 0].set_xlabel('Лаг (месяцы)')
axes[1, 0].set_ylabel('Автокорреляция')
axes[1, 0].legend(fontsize=8)

# PACF дифференцированного ряда (d=2)
plot_pacf(depos_train_diff, lags=36, ax=axes[1, 1])
axes[1, 1].axhline(y=threshold, color='red', linestyle='--', linewidth=1, alpha=0.7, label=f'±1.96/√n')
axes[1, 1].axhline(y=-threshold, color='red', linestyle='--', linewidth=1, alpha=0.7)
axes[1, 1].set_title('PACF: Дифференцированный ряд (d=2)')
axes[1, 1].set_xlabel('Лаг (месяцы)')
axes[1, 1].set_ylabel('Частичная автокорреляция')
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('05_acf_pacf_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Интерпретация ACF/PACF
print("\n📌 Наблюдения из ACF/PACF:")
print("""
   ИСХОДНЫЙ РЯД (d=0):
   - ACF: медленно убывающая → нестационарность
   - PACF: значимый пик только на лаге 1

   ДИФФЕРЕНЦИРОВАННЫЙ РЯД (d=2):
   - ACF: значимые лаги 1, 11, 13, 23, 24...
   - PACF: значимые лаги 1, 2, 3, 4, 11

   ВЫВОД: Наличие значимых лагов 11, 13, 23, 24 в ACF указывает
   на ГОДОВУЮ сезонность (s=12). Квартальная сезонность (s=4)
   не подтверждается.
""")

# ============================================================
# 7. АВТОМАТИЧЕСКИЙ ПОДБОР ПАРАМЕТРОВ SARIMA
# ============================================================

print("\n" + "="*60)
print("7. АВТОМАТИЧЕСКИЙ ПОДБОР ПАРАМЕТРОВ SARIMA")
print("="*60)

def evaluate_sarima_model(order, seasonal_order, train_data, test_data):
    """
    Обучает SARIMA модель и оценивает качество + диагностику остатков.
    """
    try:
        model = SARIMAX(
            train_data,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False)

        # Прогноз
        forecast = results.forecast(steps=len(test_data))

        # Метрики на тесте
        r2_test = r2_score(test_data, forecast)
        rmse_test = np.sqrt(mean_squared_error(test_data, forecast))
        mae_test = mean_absolute_error(test_data, forecast)

        # Диагностика остатков (Ljung-Box)
        residuals = results.resid
        lb_test = acorr_ljungbox(residuals, lags=[4, 8, 12], return_df=True)
        lb_min_p = lb_test['lb_pvalue'].min()

        # Проверка: остатки не автокоррелированы?
        residuals_ok = lb_min_p > 0.05

        return {
            'model': results,
            'forecast': forecast,
            'r2_test': r2_test,
            'rmse_test': rmse_test,
            'mae_test': mae_test,
            'aic': results.aic,
            'bic': results.bic,
            'lb_min_p': lb_min_p,
            'residuals_ok': residuals_ok,
            'success': True
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

# Определяем сетку параметров
# Учитываем годовую сезонность s=12 (из ACF/PACF)
p_range = range(0, 4)  # 0-3
d = d_order  # = 2
q_range = range(0, 4)  # 0-3
P_range = range(0, 2)  # 0-1
D_range = range(0, 2)  # 0-1
Q_range = range(0, 2)  # 0-1
s = 12  # ГОДОВАЯ сезонность (из ACF/PACF)

print(f"\n🔍 Запуск grid search...")
print(f"   Несезонные: p={list(p_range)}, d={[d]}, q={list(q_range)}")
print(f"   Сезонные: P={list(P_range)}, D={list(D_range)}, Q={list(Q_range)}, s={s}")
print(f"   Всего комбинаций: {len(p_range) * len(q_range) * len(P_range) * len(D_range) * len(Q_range)}")
print(f"\n⌛ Поиск моделей с некоррелированными остатками может занять несколько минут...")

all_results = []

for p, q, P, D, Q in product(p_range, q_range, P_range, D_range, Q_range):
    order = (p, d, q)
    seasonal_order = (P, D, Q, s)

    result = evaluate_sarima_model(order, seasonal_order, depos_train, depos_test)

    if result['success']:
        all_results.append({
            'order': order,
            'seasonal_order': seasonal_order,
            'r2_test': result['r2_test'],
            'rmse_test': result['rmse_test'],
            'mae_test': result['mae_test'],
            'aic': result['aic'],
            'bic': result['bic'],
            'lb_min_p': result['lb_min_p'],
            'residuals_ok': result['residuals_ok'],
            'result': result
        })

print(f"\n📊 Grid search завершен. Найдено {len(all_results)} моделей.")

# Модели с "хорошими" остатками (Ljung-Box p > 0.05)
good_residuals = [m for m in all_results if m['residuals_ok']]
print(f"\n📊 Моделей с некоррелированными остатками (Ljung-Box p > 0.05): {len(good_residuals)}")

if good_residuals:
    print("\n🏆 Модели с некоррелированными остатками (по AIC):")
    good_by_aic = sorted(good_residuals, key=lambda x: x['aic'])
    for i, model in enumerate(good_by_aic[:5], 1):
        print(f"   {i}. SARIMA{model['order']}×{model['seasonal_order']}: "
              f"AIC={model['aic']:.2f}, R²_test={model['r2_test']:.4f}, "
              f"LB p={model['lb_min_p']:.4f}")

    best_model_info = good_by_aic[0]
    print(f"\n✅ ВЫБОР: Модель с некоррелированными остатками и минимальным AIC")
else:
    print("\n⚠️ Ни одна модель не дала некоррелированных остатков!")
    print("   → Выбираем модель с максимальным p-value Ljung-Box (наименее автокоррелированные остатки)")

    best_by_lb = max(all_results, key=lambda x: x['lb_min_p'])
    print(f"   → Модель: SARIMA{best_by_lb['order']}×{best_by_lb['seasonal_order']}, "
          f"LB p={best_by_lb['lb_min_p']:.4f}")
    best_model_info = best_by_lb

best_order = best_model_info['order']
best_seasonal_order = best_model_info['seasonal_order']
best_result = best_model_info['result']

print(f"\n✅ Выбранная модель: SARIMA{best_order}×{best_seasonal_order}")
print(f"   AIC = {best_model_info['aic']:.2f}")
print(f"   BIC = {best_model_info['bic']:.2f}")
print(f"   R²_test = {best_model_info['r2_test']:.4f}")
print(f"   RMSE_test = {best_model_info['rmse_test']:.2f} млрд руб.")
print(f"   Ljung-Box min p = {best_model_info['lb_min_p']:.4f}")
print(f"   Остатки: {'✅ Некоррелированы' if best_model_info['residuals_ok'] else '⚠️ Автокоррелированы'}")

# ============================================================
# 8. ОБУЧЕНИЕ ВЫБРАННОЙ МОДЕЛИ И ДИАГНОСТИКА
# ============================================================

print("\n" + "="*60)
print("8. ОБУЧЕНИЕ ВЫБРАННОЙ МОДЕЛИ И ДИАГНОСТИКА")
print("="*60)

print(f"\n🔧 Обучение SARIMA{best_order}×{best_seasonal_order}...")

sarima_model = SARIMAX(
    depos_train,
    order=best_order,
    seasonal_order=best_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_results = sarima_model.fit(disp=False)

print("✅ Модель обучена")

# 8.1. Сводка модели
print("\n📊 Сводка модели:")
print(sarima_results.summary())

# 8.2. Диагностика остатков
print("\n🔍 Диагностика остатков SARIMA:")

residuals_sarima = sarima_results.resid

# Ljung-Box тест
lb_test = acorr_ljungbox(residuals_sarima, lags=[4, 8, 12], return_df=True)
print("\n📊 Ljung-Box тест (автокорреляция):")
print(lb_test.round(4))

lb_min_p = lb_test['lb_pvalue'].min()
if lb_min_p > 0.05:
    print(f"   Вывод: ✅ Нет автокорреляции (все p > 0.05)")
else:
    print(f"   Вывод: ⚠️ Автокорреляция осталась (минимальный p = {lb_min_p:.4f})")
    print(f"   → Значимые лаги требуют дополнительных AR/MA компонент")
    print(f"   → Рекомендуется итеративное добавление параметров")

# Графики диагностики
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Диагностика SARIMA{best_order}×{best_seasonal_order}', fontsize=14)

axes[0, 0].plot(residuals_sarima.index, residuals_sarima, linewidth=1)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=1)
axes[0, 0].set_title('Остатки модели')
axes[0, 0].set_xlabel('Дата')
axes[0, 0].set_ylabel('Остатки')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(residuals_sarima, bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=1)
axes[0, 1].set_title('Распределение остатков')
axes[0, 1].set_xlabel('Остатки')
axes[0, 1].set_ylabel('Частота')
axes[0, 1].grid(True, alpha=0.3)

plot_acf(residuals_sarima, lags=24, ax=axes[1, 0])
axes[1, 0].set_title('ACF остатков')
axes[1, 0].set_xlabel('Лаг')
axes[1, 0].set_ylabel('Автокорреляция')

from scipy import stats
stats.probplot(residuals_sarima, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plot')

plt.tight_layout()
plt.savefig('05_sarima_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Диагностика завершена")

# ============================================================
# 9. ПРОГНОЗ И СРАВНЕНИЕ
# ============================================================

print("\n" + "="*60)
print("9. ПРОГНОЗ И СРАВНЕНИЕ")
print("="*60)

forecast_test = sarima_results.forecast(steps=len(depos_test))
fitted_values = sarima_results.fittedvalues

r2_train_sarima = r2_score(depos_train, fitted_values)
rmse_train_sarima = np.sqrt(mean_squared_error(depos_train, fitted_values))

r2_test_sarima = r2_score(depos_test, forecast_test)
rmse_test_sarima = np.sqrt(mean_squared_error(depos_test, forecast_test))
mae_test_sarima = mean_absolute_error(depos_test, forecast_test)

print(f"\n📊 SARIMA{best_order}×{best_seasonal_order}:")
print(f"   ОБУЧАЮЩАЯ выборка (n={len(depos_train)}):")
print(f"     R²_train = {r2_train_sarima:.4f}")
print(f"     RMSE_train = {rmse_train_sarima:.2f} млрд руб.")
print(f"   ТЕСТОВАЯ выборка (n={len(depos_test)}):")
print(f"     R²_test = {r2_test_sarima:.4f}")
print(f"     RMSE_test = {rmse_test_sarima:.2f} млрд руб.")
print(f"     MAE_test = {mae_test_sarima:.2f} млрд руб.")

print(f"\n📊 Сравнение с Ridge-моделью (38 признаков):")
print(f"   {'Модель':<30} {'R²_test':<10} {'RMSE_test':<12} {'MAE_test':<12}")
print(f"   {'-'*65}")
print(f"   {'Ridge (38 признаков)':<30} {'0.9422':<10} {'566.09':<12} {'489.89':<12}")
print(f"   {f'SARIMA{best_order}×{best_seasonal_order}':<30} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {mae_test_sarima:<12.2f}")

# Визуализация прогноза
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['DEPOS'], label='Фактические данные', color='#1f77b4', linewidth=2.5)
plt.plot(depos_test.index, forecast_test, label=f'Прогноз SARIMA{best_order}×{best_seasonal_order}',
         color='#ff7f0e', linestyle='--', linewidth=2.5)

forecast_result = sarima_results.get_forecast(steps=len(depos_test))
confidence_intervals = forecast_result.conf_int(alpha=0.05)

plt.fill_between(
    depos_test.index,
    confidence_intervals.iloc[:, 0],
    confidence_intervals.iloc[:, 1],
    alpha=0.25, color='#ff7f0e', label='95% доверительный интервал'
)

plt.title(f'Прогноз объема вкладов — SARIMA{best_order}×{best_seasonal_order}\n' +
          f'R²_test = {r2_test_sarima:.4f}, RMSE = {rmse_test_sarima:.2f} млрд руб.',
          fontsize=14)
plt.xlabel('Дата')
plt.ylabel('Объем вкладов, млрд руб.')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('05_sarima_forecast.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 10. SARIMAX С ЭКЗОГЕННЫМИ ПЕРЕМЕННЫМИ
# ============================================================

print("\n" + "="*60)
print("10. SARIMAX С ЭКЗОГЕННЫМИ ПЕРЕМЕННЫМИ")
print("="*60)

print("""
📌 ИДЕЯ: Добавить макроэкономические показатели как экзогенные переменные.
Это позволит модели учитывать внешние факторы при прогнозировании.

Преимущества:
- Учитывает влияние WAGE, CPI, UNEM и других факторов
- Может лучше адаптироваться к структурным изменениям
- Комбинирует сильные стороны SARIMA и Ridge
""")

# Подготавливаем экзогенные переменные
# Используем те же признаки, что и в Ridge-модели (блок 4)
exog_features = ['WAGE', 'SERV', 'DEP1', 'CRED1', 'CPI', 'USDind', 'UNEM', 'IPI', 'IMP']

# Создаем лаги для экзогенных переменных
for col in exog_features:
    for lag in [1, 3, 6]:
        df[f'{col}_lag_{lag}'] = df[col].shift(lag)

# Добавляем сезонные фиктивные переменные
df['Month'] = df.index.month
for month in range(2, 13):
    df[f'month_{month}'] = (df['Month'] == month).astype(int)

# Добавляем структурный разрыв post_2022
df['post_2022'] = (df.index >= '2023-01-01').astype(int)

# Формируем полный набор экзогенных переменных
exog_columns = [col for col in df.columns if col not in [
    'DEPOS', 'DEPOS_log', 'DEPOS_diff1', 'DEPOS_diff2', 'Month'
]]

print(f"\n📊 Всего экзогенных переменных: {len(exog_columns)}")
print(f"   Базовые: {len(exog_features)}")
print(f"   Лаги: {len(exog_features) * 3}")
print(f"   Сезонные: 11")
print(f"   Структурные: 1 (post_2022)")

# Подготовка train/test для экзогенных переменных
exog_full = df[exog_columns].dropna()
common_index = depos_train.index.intersection(exog_full.index)

exog_train = exog_full.loc[common_index]
depos_train_aligned = depos_train.loc[common_index]

# Для тестового периода берем последние 12 записей
exog_test = exog_full.iloc[-12:]

print(f"\n📊 Обучающая выборка (с экзогенными): {len(depos_train_aligned)} записей")
print(f"📊 Тестовая выборка (с экзогенными): {len(exog_test)} записей")
print(f"📊 Период обучения: {depos_train_aligned.index[0]} — {depos_train_aligned.index[-1]}")
print(f"📊 Период теста: {exog_test.index[0]} — {exog_test.index[-1]}")

# Обучение SARIMAX с экзогенными переменными
print(f"\n🔧 Обучение SARIMAX{best_order}×{best_seasonal_order} с экзогенными...")
print(f"   Это может занять около минуты...")

try:
    sarimax_model = SARIMAX(
        depos_train_aligned,
        exog=exog_train,
        order=best_order,
        seasonal_order=best_seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    sarimax_results = sarimax_model.fit(disp=False, maxiter=200)

    print("✅ Модель SARIMAX обучена")

    # Прогноз на тестовый период
    forecast_sarimax = sarimax_results.forecast(steps=len(depos_test), exog=exog_test)

    # Метрики на тестовой выборке
    r2_test_sarimax = r2_score(depos_test, forecast_sarimax)
    rmse_test_sarimax = np.sqrt(mean_squared_error(depos_test, forecast_sarimax))
    mae_test_sarimax = mean_absolute_error(depos_test, forecast_sarimax)

    # Метрики на обучающей выборке
    fitted_sarimax = sarimax_results.fittedvalues
    r2_train_sarimax = r2_score(depos_train_aligned, fitted_sarimax)
    rmse_train_sarimax = np.sqrt(mean_squared_error(depos_train_aligned, fitted_sarimax))

    print(f"\n📊 SARIMAX с экзогенными переменными:")
    print(f"   ОБУЧАЮЩАЯ выборка (n={len(depos_train_aligned)}):")
    print(f"     R²_train = {r2_train_sarimax:.4f}")
    print(f"     RMSE_train = {rmse_train_sarimax:.2f} млрд руб.")
    print(f"   ТЕСТОВАЯ выборка (n={len(depos_test)}):")
    print(f"     R²_test = {r2_test_sarimax:.4f}")
    print(f"     RMSE_test = {rmse_test_sarimax:.2f} млрд руб.")
    print(f"     MAE_test = {mae_test_sarimax:.2f} млрд руб.")

    # Диагностика остатков SARIMAX
    residuals_sarimax = sarimax_results.resid
    lb_test_sarimax = acorr_ljungbox(residuals_sarimax, lags=[4, 8, 12], return_df=True)
    lb_min_p_sarimax = lb_test_sarimax['lb_pvalue'].min()

    print(f"\n   ДИАГНОСТИКА ОСТАТКОВ:")
    print(f"   Ljung-Box min p = {lb_min_p_sarimax:.4f}")
    print(f"   Остатки: {'✅ Некоррелированы' if lb_min_p_sarimax > 0.05 else '⚠️ Автокоррелированы'}")

    # Сравнение всех моделей
    print(f"\n📊 ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ:")
    print(f"   {'Модель':<35} {'R²_test':<10} {'RMSE':<12} {'MAE':<12} {'LB p':<10}")
    print(f"   {'-'*80}")
    print(f"   {'SARIMA (без экзогенных)':<35} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {mae_test_sarima:<12.2f} {lb_min_p:<10.4f}")
    print(f"   {'SARIMAX (с экзогенными)':<35} {r2_test_sarimax:<10.4f} {rmse_test_sarimax:<12.2f} {mae_test_sarimax:<12.2f} {lb_min_p_sarimax:<10.4f}")
    print(f"   {'Ridge (38 признаков, блок 4)':<35} {'0.9422':<10} {'566.09':<12} {'489.89':<12} {'—':<10}")

    # Визуализация прогнозов
    plt.figure(figsize=(14, 7))

    plt.plot(df.index, df['DEPOS'], label='Фактические данные', color='#1f77b4', linewidth=2.5)
    plt.plot(depos_test.index, forecast_test, label=f'SARIMA{best_order}×{best_seasonal_order}',
             color='#ff7f0e', linestyle='--', linewidth=2)
    plt.plot(depos_test.index, forecast_sarimax, label='SARIMAX (с экзогенными)',
             color='#2ca02c', linestyle='--', linewidth=2)

    plt.title('Сравнение прогнозов: SARIMA vs SARIMAX', fontsize=14)
    plt.xlabel('Дата')
    plt.ylabel('Объем вкладов, млрд руб.')
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('05_sarimax_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Визуальный анализ прогнозов
    print("\n📊 ВИЗУАЛЬНЫЙ АНАЛИЗ ПРОГНОЗОВ:")
    print(f"   Фактические значения (тест): {depos_test.min():.0f} — {depos_test.max():.0f} млрд руб.")
    print(f"   SARIMA прогноз: {forecast_test.min():.0f} — {forecast_test.max():.0f} млрд руб.")
    print(f"   SARIMAX прогноз: {forecast_sarimax.min():.0f} — {forecast_sarimax.max():.0f} млрд руб.")

    # Проверка завышения/занижения
    mean_actual = depos_test.mean()
    mean_sarima = forecast_test.mean()
    mean_sarimax = forecast_sarimax.mean()

    print(f"\n   Средние значения:")
    print(f"   Фактические: {mean_actual:.0f} млрд руб.")
    print(f"   SARIMA: {mean_sarima:.0f} млрд руб. ({(mean_sarima - mean_actual) / mean_actual * 100:+.1f}%)")
    print(f"   SARIMAX: {mean_sarimax:.0f} млрд руб. ({(mean_sarimax - mean_actual) / mean_actual * 100:+.1f}%)")

    if abs(mean_sarimax - mean_actual) > abs(mean_sarima - mean_actual):
        print(f"\n   ⚠️ SARIMAX дает {'ЗАВЫШЕННЫЕ' if mean_sarimax > mean_actual else 'ЗАНИЖЕННЫЕ'} прогнозы")
        print(f"   → Экзогенные переменные искажают прогноз из-за переобучения")
        print(f"   → 48 экзогенных переменных слишком много для n=128")
    else:
        print(f"\n   ✅ SARIMAX прогнозы ближе к фактическим, чем SARIMA")

    # Сохраняем результаты SARIMAX
    sarimax_metrics = {
        'r2_train': r2_train_sarimax,
        'rmse_train': rmse_train_sarimax,
        'r2_test': r2_test_sarimax,
        'rmse_test': rmse_test_sarimax,
        'mae_test': mae_test_sarimax,
        'lb_min_p': lb_min_p_sarimax,
        'forecast': forecast_sarimax
    }

    print(f"\n✅ SARIMAX анализ завершен")
    print(f"   → SARIMAX не улучшает прогноз из-за переобучения на 48 экзогенных переменных")
    print(f"   → Ridge (38 признаков) остается лучшей моделью")

except Exception as e:
    print(f"⚠️ Ошибка в SARIMAX: {e}")
    print("   Возможные причины:")
    print("   - Мультиколлинеарность экзогенных переменных")
    print("   - Недостаточно степеней свободы (слишком много параметров)")
    print("   - Проблемы сходимости")
    print("   → Рекомендуется уменьшить число экзогенных переменных")
    print("   → Или использовать только ключевые: WAGE, CPI, UNEM, DEP1")
    sarimax_metrics = None

# ============================================================
# 11. КОМБИНИРОВАННАЯ МОДЕЛЬ: SARIMA + Ridge
# ============================================================

print("\n" + "="*60)
print("11. КОМБИНИРОВАННАЯ МОДЕЛЬ: SARIMA + Ridge")
print("="*60)

print("""
📌 ИДЕЯ: Объединить сильные стороны обеих моделей.

Ridge прогнозирует тренд, SARIMA корректирует остатки
  Шаг 1: Ridge (38 признаков) прогнозирует DEPOS
  Шаг 2: Вычисляем остатки: e = DEPOS_actual - DEPOS_ridge
  Шаг 3: SARIMA моделирует остатки
  Шаг 4: Итоговый прогноз = Ridge + SARIMA(остатки)
""")

print("\n" + "-"*60)
print("Ridge прогнозирует, SARIMA корректирует остатки")
print("-"*60)

# Шаг 1: Пересчитываем Ridge-модель (38 признаков из блока 4)
print("\n🔧 Шаг 1: Обучение Ridge-модели...")

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Восстанавливаем признаки из блока 4
df_ridge = df.copy()

# Лаги DEPOS
df_ridge['DEPOS_log'] = np.log(df_ridge['DEPOS'])
for lag in [1, 3, 6, 12]:
    df_ridge[f'DEPOS_lag_{lag}'] = df_ridge['DEPOS'].shift(lag)

# Взаимодействие UNEM × DEP1
df_ridge['UNEM_DEP1'] = df_ridge['UNEM'] * df_ridge['DEP1']

# Аномалии WAGE
from sklearn.linear_model import LinearRegression
model_wage = LinearRegression()
model_wage.fit(df_ridge[['WAGE']].values, df_ridge['DEPOS'].values)
df_ridge['residual_wage'] = df_ridge['DEPOS'] - model_wage.predict(df_ridge[['WAGE']].values)
threshold_anomaly = -2.0 * df_ridge['residual_wage'].std()
df_ridge['anomaly_wage'] = (df_ridge['residual_wage'] < threshold_anomaly).astype(int)

# Формируем X и y для Ridge
X_ridge = df_ridge.drop(['DEPOS', 'DEPOS_log', 'Month', 'residual_wage'], axis=1).dropna()
y_ridge = df_ridge.loc[X_ridge.index, 'DEPOS']

# Разделение train/test
train_size_ridge = len(X_ridge) - 12
X_train_ridge = X_ridge.iloc[:train_size_ridge]
X_test_ridge = X_ridge.iloc[train_size_ridge:]
y_train_ridge = y_ridge.iloc[:train_size_ridge]
y_test_ridge = y_ridge.iloc[train_size_ridge:]

# Масштабирование
scaler_ridge = StandardScaler()
X_train_scaled_ridge = scaler_ridge.fit_transform(X_train_ridge)
X_test_scaled_ridge = scaler_ridge.transform(X_test_ridge)

# Обучение Ridge
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled_ridge, y_train_ridge)

# Прогнозы Ridge
ridge_train_pred = ridge_model.predict(X_train_scaled_ridge)
ridge_test_pred = ridge_model.predict(X_test_scaled_ridge)

print(f"   ✅ Ridge обучена: {X_ridge.shape[1]} признаков")
print(f"   R²_train = {r2_score(y_train_ridge, ridge_train_pred):.4f}")
print(f"   R²_test = {r2_score(y_test_ridge, ridge_test_pred):.4f}")

# Шаг 2: Вычисляем остатки Ridge
print("\n🔧 Шаг 2: Вычисление остатков Ridge...")

residuals_ridge_train = y_train_ridge.values - ridge_train_pred
residuals_ridge_test = y_test_ridge.values - ridge_test_pred

print(f"   Остатки на обучающей выборке: {len(residuals_ridge_train)}")
print(f"   Остатки на тестовой выборке: {len(residuals_ridge_test)}")

# Шаг 3: SARIMA моделирует остатки Ridge
print("\n🔧 Шаг 3: SARIMA моделирует остатки Ridge...")
print("   Подбор оптимальной SARIMA для остатков...")
print("   Это может занять около минуты...")

# Преобразуем остатки в Series с датами
residuals_ridge_train_series = pd.Series(
    residuals_ridge_train,
    index=y_train_ridge.index
)

# Подбираем SARIMA для остатков
best_residual_model = None
best_residual_aic = np.inf

for p in range(0, 3):
    for q in range(0, 3):
        for d_resid in range(0, 2):
            try:
                model_resid = SARIMAX(
                    residuals_ridge_train_series,
                    order=(p, d_resid, q),
                    seasonal_order=(0, 0, 0, 0),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                results_resid = model_resid.fit(disp=False)

                if results_resid.aic < best_residual_aic:
                    best_residual_aic = results_resid.aic
                    best_residual_model = results_resid
                    best_residual_order = (p, d_resid, q)
            except:
                continue

if best_residual_model is not None:
    print(f"   ✅ Лучшая SARIMA для остатков: {best_residual_order}")
    print(f"   AIC = {best_residual_aic:.2f}")

    # Прогноз остатков
    residual_forecast = best_residual_model.forecast(steps=len(y_test_ridge))

    # Шаг 4: Итоговый прогноз
    print("\n🔧 Шаг 4: Комбинированный прогноз...")

    combined_forecast = ridge_test_pred + residual_forecast.values

    # Метрики комбинированной модели
    r2_combined = r2_score(y_test_ridge, combined_forecast)
    rmse_combined = np.sqrt(mean_squared_error(y_test_ridge, combined_forecast))
    mae_combined = mean_absolute_error(y_test_ridge, combined_forecast)

    print(f"\n📊 Комбинированная модель (Ridge + SARIMA):")
    print(f"   ОБУЧАЮЩАЯ выборка:")
    print(f"     R²_train = {r2_score(y_train_ridge, ridge_train_pred + best_residual_model.fittedvalues):.4f}")
    print(f"   ТЕСТОВАЯ выборка:")
    print(f"     R²_test = {r2_combined:.4f}")
    print(f"     RMSE_test = {rmse_combined:.2f} млрд руб.")
    print(f"     MAE_test = {mae_combined:.2f} млрд руб.")

    # Диагностика остатков (исправлено)
    combined_residuals = y_test_ridge.values - combined_forecast

    print(f"\n   ДИАГНОСТИКА ОСТАТКОВ (тестовая выборка, n={len(combined_residuals)}):")

    try:
        # Для n=12 используем лаги [1, 2, 3, 4]
        lb_combined = acorr_ljungbox(combined_residuals, lags=[1, 2, 3, 4], return_df=True)
        lb_combined_min_p = lb_combined['lb_pvalue'].min()

        print(f"   Ljung-Box min p = {lb_combined_min_p:.4f}")
        print(f"   Остатки: {'✅ Некоррелированы' if lb_combined_min_p > 0.05 else '⚠️ Автокоррелированы'}")
    except Exception as e:
        print(f"   ⚠️ Ljung-Box тест не применим: {e}")
        lb_combined_min_p = None

    # Итоговое сравнение
    print(f"\n📊 ИТОГОВОЕ СРАВНЕНИЕ ВСЕХ МОДЕЛЕЙ:")
    print(f"   {'Модель':<40} {'R²_test':<10} {'RMSE':<12} {'MAE':<12}")
    print(f"   {'-'*75}")
    print(f"   {'SARIMA (без экзогенных)':<40} {r2_test_sarima:<10.4f} {rmse_test_sarima:<12.2f} {mae_test_sarima:<12.2f}")

    if sarimax_metrics is not None:
        print(f"   {'SARIMAX (с экзогенными)':<40} {sarimax_metrics['r2_test']:<10.4f} {sarimax_metrics['rmse_test']:<12.2f} {sarimax_metrics['mae_test']:<12.2f}")

    print(f"   {'Ridge':<40} {r2_score(y_test_ridge, ridge_test_pred):<10.4f} {np.sqrt(mean_squared_error(y_test_ridge, ridge_test_pred)):<12.2f} {mean_absolute_error(y_test_ridge, ridge_test_pred):<12.2f}")
    print(f"   {'Комбинированная (Ridge + SARIMA)':<40} {r2_combined:<10.4f} {rmse_combined:<12.2f} {mae_combined:<12.2f}")

    # Визуализация
    plt.figure(figsize=(14, 7))

    plt.plot(df.index, df['DEPOS'], label='Фактические данные', color='#1f77b4', linewidth=2.5)
    plt.plot(y_test_ridge.index, ridge_test_pred, label='Ridge',
             color='#ff7f0e', linestyle='--', linewidth=2)
    plt.plot(y_test_ridge.index, combined_forecast, label='Комбинированная (Ridge + SARIMA)',
             color='#2ca02c', linestyle='--', linewidth=2)

    plt.title('Сравнение прогнозов: Ridge vs Комбинированная модель', fontsize=14)
    plt.xlabel('Дата')
    plt.ylabel('Объем вкладов, млрд руб.')
    plt.legend(loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('05_combined_model_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Визуальный анализ
    print("\n📊 ВИЗУАЛЬНЫЙ АНАЛИЗ КОМБИНИРОВАННОЙ МОДЕЛИ:")

    mean_actual = y_test_ridge.mean()
    mean_ridge = ridge_test_pred.mean()
    mean_combined = combined_forecast.mean()

    print(f"   Средние значения (тест):")
    print(f"   Фактические: {mean_actual:.0f} млрд руб.")
    print(f"   Ridge: {mean_ridge:.0f} млрд руб. ({(mean_ridge - mean_actual) / mean_actual * 100:+.1f}%)")
    print(f"   Комбинированная: {mean_combined:.0f} млрд руб. ({(mean_combined - mean_actual) / mean_actual * 100:+.1f}%)")

    ridge_errors = np.abs(y_test_ridge.values - ridge_test_pred)
    combined_errors = np.abs(y_test_ridge.values - combined_forecast)

    print(f"\n   Средняя абсолютная ошибка:")
    print(f"   Ridge: {ridge_errors.mean():.0f} млрд руб.")
    print(f"   Комбинированная: {combined_errors.mean():.0f} млрд руб.")

    if combined_errors.mean() < ridge_errors.mean():
        print(f"\n   ✅ Комбинированная модель УЛУЧШАЕТ прогноз Ridge")
        print(f"   Улучшение: {(ridge_errors.mean() - combined_errors.mean()) / ridge_errors.mean() * 100:.1f}%")
    else:
        print(f"\n   ⚠️ Комбинированная модель НЕ улучшает прогноз Ridge")
        print(f"   Ухудшение: {(combined_errors.mean() - ridge_errors.mean()) / ridge_errors.mean() * 100:.1f}%")

else:
    print("⚠️ Не удалось подобрать SARIMA для остатков Ridge")

# ============================================================
# 12. ИТОГОВЫЙ ВЫВОД ПО БЛОКУ 5
# ============================================================

print("\n" + "="*60)
print("12. ИТОГОВЫЙ ВЫВОД ПО БЛОКУ 5")
print("="*60)

print(f"""
📌 КЛЮЧЕВЫЕ РЕЗУЛЬТАТЫ БЛОКА 5 (ПОЛНЫЙ ОБЗОР):

1. СТАЦИОНАРНОСТЬ:
   - Исходный ряд DEPOS: нестационарен (d=2)
   - Годовая сезонность (s=12) подтверждена ACF/PACF

2. МОДЕЛИ:
   ├── SARIMA{best_order}×{best_seasonal_order}:
   │   R²_test = {r2_test_sarima:.4f}, RMSE = {rmse_test_sarima:.2f}
   │   Остатки: ✅ Некоррелированы (LB p = {lb_min_p:.4f})
   │
   ├── SARIMAX (48 экзогенных):
""")

if sarimax_metrics is not None:
    print(f"""   │   R²_test = {sarimax_metrics['r2_test']:.4f}, RMSE = {sarimax_metrics['rmse_test']:.2f}
   │   Остатки: {'✅ Некоррелированы' if sarimax_metrics['lb_min_p'] > 0.05 else '⚠️ Автокоррелированы'}
   │   Проблема: переобучение на 48 переменных
""")
else:
    print(f"""   │   Не удалось обучить (ошибка)
""")

print(f"""   ├── Ridge:
   │   R²_test = {r2_score(y_test_ridge, ridge_test_pred):.4f}, RMSE = {np.sqrt(mean_squared_error(y_test_ridge, ridge_test_pred)):.2f}
   │
   └── Комбинированная (Ridge + SARIMA):
       R²_test = {r2_combined:.4f}, RMSE = {rmse_combined:.2f}

3. ГЛАВНЫЕ ВЫВОДЫ:
   - SARIMA решает проблему автокорреляции ✅
   - SARIMAX переобучается из-за избытка экзогенных переменных ❌
   - Ridge — лучшая модель для прогноза ✅
   - Комбинированная модель не улучшает Ridge ❌

4. РЕКОМЕНДАЦИИ:
   - Для прогнозирования: Ridge (R²_test = {r2_score(y_test_ridge, ridge_test_pred):.4f})
   - Для понимания временной структуры: SARIMA (остатки некоррелированы)
   - SARIMAX требует сокращения экзогенных переменных (5-10 ключевых)
   - Комбинированный подход не дает улучшения на данном этапе

5. ИТОГОВОЕ РЕШЕНИЕ:
   - БЛОК 4 (Ridge) — финальная модель для прогноза
   - БЛОК 5 (SARIMA) — диагностический инструмент
""")

print("✅ БЛОК 5 ПОЛНОСТЬЮ ЗАВЕРШЕН")